### 1. Read and prepare the cleaned dataset ###

In [1]:
### Import necessary libraries ###
import time
import pandas as pd
import numpy as np
import tqdm.auto as tqdm
import os

import helper_func as hf

from functools import lru_cache

import pubchempy as pcp

from rdkit import Chem
from descriptastorus.descriptors import rdNormalizedDescriptors

C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = './Cleaned_VLE_Data.csv'
df = pd.read_csv(data_path)
print(df.head())

                      property    value phase  Temperature, K  Pressure, kPa  \
0  Thermal conductivity, W/m/K  0.02096   Gas           300.0          724.0   
1  Thermal conductivity, W/m/K  0.02125   Gas           300.0         1288.0   
2  Thermal conductivity, W/m/K  0.02161   Gas           300.0         1835.0   
3  Thermal conductivity, W/m/K  0.02197   Gas           300.0         2335.0   
4  Thermal conductivity, W/m/K  0.02245   Gas           300.0         2820.0   

   Mole fraction     Component 1 Component 2  
0         0.2493  carbon dioxide     methane  
1         0.2493  carbon dioxide     methane  
2         0.2493  carbon dioxide     methane  
3         0.2493  carbon dioxide     methane  
4         0.2493  carbon dioxide     methane  


In [3]:
df_oh, cat_map = hf.one_hot_encode(df.drop(columns=['Component 1', 'Component 2']), drop_first=True, dummy_na=False, prefix_sep="_")
df_oh['Component 1'] = df['Component 1']
df_oh['Component 2'] = df['Component 2']

In [4]:
@lru_cache(maxsize=None)
def get_smiles(name, retries=3):
    for i in range(retries):
        try:
            compounds = pcp.get_compounds(name, 'name')
            if compounds:
                return compounds[0].isomeric_smiles
            return None
        except Exception as e:
            if "SSL" in str(e) or "EOF" in str(e):
                print(f"SSL Error for {name}, retrying ({i+1}/{retries})...")
                time.sleep(2) # Give the connection a moment to breathe
                continue
            print(f"Permanent Error fetching {name}: {e}")
            break
    return None

# Apply the cached function to get SMILES for both components
df_oh['Smiles 1'] = df_oh['Component 1'].apply(get_smiles)
df_oh['Smiles 2'] = df_oh['Component 2'].apply(get_smiles)

# Replace empty strings with NaN across the specific columns
df_oh['Smiles 1'] = df_oh['Smiles 1'].replace('', np.nan)
df_oh['Smiles 2'] = df_oh['Smiles 2'].replace('', np.nan)

# Check for empty/NaN values in 'Smiles 1' and 'Smiles 2'
empty_count = (df_oh['Smiles 1'].isnull() | df_oh['Smiles 2'].isnull()).sum()
print(f"Number of rows with empty/NaN 'Smiles 1' or 'Smiles 2': {empty_count}")

# Drop rows with empty/NaN values in 'Smiles 1' or 'Smiles 2'
if empty_count > 0:
    df_oh = df_oh.dropna(subset=['Smiles 1', 'Smiles 2'])
    print(f"New shape after dropping: {df_oh.shape}")

df_oh['mol1'] = df_oh['Smiles 1'].apply(Chem.MolFromSmiles)
df_oh['mol2'] = df_oh['Smiles 2'].apply(Chem.MolFromSmiles)

C:\Users\admin\AppData\Local\Temp\ipykernel_7660\1703776183.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compounds[0].isomeric_smiles
C:\Users\admin\AppData\Local\Temp\ipykernel_7660\1703776183.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return compounds[0].isomeric_smiles


Number of rows with empty/NaN 'Smiles 1' or 'Smiles 2': 2241
New shape after dropping: (108142, 34)


[01:06:49] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not removing hydrogen atom without neighbors
[01:06:50] WARNING: not r

In [5]:
def get_descriptors(data_set:pd.DataFrame,
                    mol_col:list,
                    save_dir: str = "./processed_data" , #= './descriptors' Path to save the descriptor files
                    save_file: bool = True, # Whether to save the updated DataFrame with descriptors as a CSV file
                    output_df: bool = True, # Whether to return the updated DataFrame with descriptors
                    )->pd.DataFrame | None:
    """Helper function to calculate and save molecular descriptors for specified columns in a DataFrame."""

    @lru_cache(maxsize=None)
    def get_cached_descriptors(mol_input): return generator.calculateMol(mol_input, None)
    generator = rdNormalizedDescriptors.RDKit2DNormalized()

    if save_file: os.makedirs(save_dir, exist_ok=True)

    # Create a copy of the dataset to avoid modifying the original
    df = data_set.copy()
    for col in mol_col:
        print(f"Processing {col}...")
        x = np.stack([get_cached_descriptors(m) for m in tqdm.tqdm(df[col].tolist())])
        print(f"{col} shape: {x.shape}")
        if save_file: np.save(f'{save_dir}/descriptors_for_{col}.npy', x)
        df = pd.concat([df, pd.DataFrame(x, columns=[f"{col}_desc_{i}" for i in range(x.shape[1])])], axis=1)

    # Save the updated DataFrame with descriptors
    if save_file: df.to_csv(f'{save_dir}/dataset_with_descriptors.csv', index=False)
    return df if output_df else None

In [6]:
df_processed = get_descriptors(df_oh, mol_col=['mol1', 'mol2'])

Processing mol1...


 66%|██████▌   | 71470/108142 [16:05<09:03, 67.49it/s][01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
 66%|██████▌   | 71478/108142 [16:05<08:43, 70.03it/s][01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom without neighbors
[01:22:59] WARNING: not removing hydrogen atom 

mol1 shape: (108142, 200)
Processing mol2...


 98%|█████████▊| 108137/110383 [19:29<00:26, 85.39it/s] Could not compute BalabanJ for molecule
Traceback (most recent call last):
  File "C:\Users\admin\Desktop\CHE1148-VLE-Project\.venv\Lib\site-packages\rdkit\Chem\GraphDescriptors.py", line 488, in BalabanJ
    dMat = mol._balabanMat
           ^^^^^^^^^^^^^^^
AttributeError: 'float' object has no attribute '_balabanMat'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\admin\Desktop\CHE1148-VLE-Project\.venv\Lib\site-packages\descriptastorus\descriptors\rdDescriptors.py", line 432, in applyFunc
    return functions[name](m)
           ~~~~~~~~~~~~~~~^^^
  File "C:\Users\admin\Desktop\CHE1148-VLE-Project\.venv\Lib\site-packages\rdkit\Chem\GraphDescriptors.py", line 491, in BalabanJ
    dMat = Chem.GetDistanceMatrix(mol, useBO=1, useAtomWts=0, force=0, prefix="Balaban")
Boost.Python.ArgumentError: Python argument types in
    rdkit.Chem.rdmolops.GetDistanceMatrix

mol2 shape: (110383, 200)


#### 2. Train an XGBoost model ###

In [7]:
### Import necessary libraries for modeling ###
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [34]:
### Define features and target variable ###
features = [col for col in df_processed.columns if col != 'Mole fraction']
target = 'Mole fraction'
data = df_processed.dropna(subset=features)
X = data[features].drop(columns=['Component 1', 'Component 2', 'Smiles 1', 'Smiles 2', 'mol1', 'mol2'])
y = data[target]

### Split the data into training and testing sets ###
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_hold, y_hold, test_size=0.5, random_state=42)

In [38]:
### Train the XGBoost model ###
def convert_to_numeric(df):
    not_accepted = df.select_dtypes(include=['object']).columns
    for col in not_accepted:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df
def train_xgboost(X_train, y_train, X_val, y_val):
    X_train, X_val = convert_to_numeric(X_train), convert_to_numeric(X_val)
    model = XGBRegressor(n_jobs = -1, n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    model.fit(X_train, y_train)
    ### Evaluate the model on the test set ###
    y_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    print(f"Test MSE: {mse:.4f}")
    print(f"Test R^2 Score: {r2:.4f}")
    return

In [39]:
X_test.head(5)

,value,"Temperature, K","Pressure, kPa",property_Activity coefficient,"property_Amount density, mol/m3","property_Binary diffusion coefficient, m2/s",property_Compressibility factor,"property_Electrical conductivity, S/m","property_Excess molar Gibbs energy, kJ/mol","property_Excess molar enthalpy (molar enthalpy of mixing), kJ/mol",...,mol2_desc_190,mol2_desc_191,mol2_desc_192,mol2_desc_193,mol2_desc_194,mol2_desc_195,mol2_desc_196,mol2_desc_197,mol2_desc_198,mol2_desc_199
16711,0.000047,618.20,30000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.156089
104895,0.000761,393.20,60000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.192005
24043,781.400000,283.15,5000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.273923
107480,766.000000,337.40,11370.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.238586
16729,533.100000,523.20,20000.0,False,False,False,False,False,False,False,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.156089


In [40]:
train_xgboost(X_train, y_train, X_val, y_val)

Test MSE: 0.0467
Test R^2 Score: 0.4964
